In [1]:
# STEP 1 - Convert pdf into separate pages
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader

#loader = WebBaseLoader("https://en.wikipedia.org/wiki/Climate_change")
#loader = TextLoader("AI.txt")

#loader = PyPDFLoader("Artificial-Intelligence.pdf")
loader = PyPDFLoader("06-LSA Advert October 2026.pdf")

pdf_pages = loader.load()
print(f'No of pages: {len(pdf_pages)}')

C:\Users\User\AppData\Local\Temp\ipykernel_6420\70775724.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


No of pages: 2


In [2]:
# STEP 2 - Convert every page into sub-pages (chunks)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
           separators=["\n\n", "\n", ",", " "]
)
chunk_documents = splitter.split_documents(pdf_pages);
print(f'No of chunks: {len(chunk_documents)}')

No of chunks: 10


In [3]:
# STEP 3 - Convert every chunk into vectors.
# STEP 4 - Store these vectors into Vector database.
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OllamaEmbeddings(model="nomic-embed-text")
print('Embedding model is ready')

faiss_vector_store = FAISS. from_documents (chunk_documents, embedding_model)
faiss_vector_store. save_local("faiss-ai-book")
print('FAISS Vector store created')

Embedding model is ready
FAISS Vector store created


In [8]:
# STEP 5 - Build Retrieval Chain

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

model = ChatOllama(model="gpt-oss:120b-cloud")
print( 'Chat Model is ready to use!')

USER_PROMPT_TEMPLATE = """
    Use the following piece of the context to answer user's question.
    If you don't know the answer, just say you don't know, don't try to make the answer.
    Hey can you tell me the school name mentioned in this?
    ---------------------------------
    {context}
    Question: {question}
"""

prompt_template = PromptTemplate(
    template=USER_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)
qa_chain = RetrievalQA.from_chain_type(
    model,
    chain_type="stuff",
    retriever=faiss_vector_store.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={ "prompt": prompt_template }
)
print('Retrieval chain is ready!!')

Chat Model is ready to use!
Retrieval chain is ready!!


In [9]:
response = qa_chain.invoke("What is the school name for which the job application it is?")
response

{'query': 'What is the school name for which the job application it is?',
 'result': 'The job application is for **Limington House (Limington House School)**.',
 'source_documents': [Document(id='68a70c08-ad75-48dd-be16-27bcd055d73e', metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-30T09:34:26+01:00', 'author': 'lcosgro', 'moddate': '2026-09-30T09:34:26+01:00', 'source': '06-LSA Advert October 2026.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='- or contact the school office on 01256 322148. \n \nVisits to the school are warmly welcomed and encouraged. \n \nWe reserve the right to close a vacancy earlier than the advertised date if we receive applications \nthat meet the criteria. \n \nPlease note that previous applicants need not apply.'),
  Document(id='45a9b817-d6a3-416c-9373-dd929e8aaea0', metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-30T09:34:

In [10]:
response = qa_chain.invoke("Can you let me know closing date fro the application?")
response

{'query': 'Can you let me know closing date fro the application?',
 'result': 'The application closing date is **Friday\u202f23\u202fOctober\u202f2026, at\u202fNoon**.',
 'source_documents': [Document(id='68a70c08-ad75-48dd-be16-27bcd055d73e', metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-30T09:34:26+01:00', 'author': 'lcosgro', 'moddate': '2026-09-30T09:34:26+01:00', 'source': '06-LSA Advert October 2026.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='- or contact the school office on 01256 322148. \n \nVisits to the school are warmly welcomed and encouraged. \n \nWe reserve the right to close a vacancy earlier than the advertised date if we receive applications \nthat meet the criteria. \n \nPlease note that previous applicants need not apply.'),
  Document(id='59253737-d3d8-4cd5-80bf-ec3fe4a00532', metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-

In [11]:
response = qa_chain.invoke("Can you tell me no of centuries doen by Sachin Tendulkar")
response

{'query': 'Can you tell me no of centuries doen by Sachin Tendulkar',
 'result': "I don't know.",
 'source_documents': [Document(id='d94439ec-76bf-40b2-960e-0cb02aee8090', metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-30T09:34:26+01:00', 'author': 'lcosgro', 'moddate': '2026-09-30T09:34:26+01:00', 'source': '06-LSA Advert October 2026.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='• a positive ‘can do’ attitude \n• good communication and interpersonal skills \nRated a good school (OFSTED, April 2025). \n“Pupils’ lives are transformed by attending this school”. \n“Pupils’ engagement and attention in lessons is exemplary”.'),
  Document(id='45a9b817-d6a3-416c-9373-dd929e8aaea0', metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-30T09:34:26+01:00', 'author': 'lcosgro', 'moddate': '2026-09-30T09:34:26+01:00', 'source': '06-LSA Advert October 2026.pdf', '

In [13]:
# STEP 6: Build UI interface

import gradio as gr

def chatbot_response(message, history):
# Use the created RetrievalQA chain to get the answer
response = qa_chain({"query": message})
answer = response["result"]
source_documents = response["source_documents"]

# Format the redponse to include the answer and source documents (optional)
formatted_response = f"{answer}" # You can add source documents here if desired

print(f"Source documents: {source_documents}")

return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
fn=chatbot_response,
title="RAG AI Chatbot",
description="Ask questions about A."
)

# Launch the interface
iface.launch(share=True, server_port=7861)

IndentationError: expected an indented block after function definition on line 5 (1981469457.py, line 7)